<a href="https://colab.research.google.com/github/DL-ResearchSpace/EEG-_Federated_Classification/blob/main/FedFIR_GAN_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Step 1. Connect Google Drive and unpack the code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/fedfir_work'      # everything important is saved here
os.makedirs(WORK, exist_ok=True)
!cp /content/drive/MyDrive/FedFIR_GAN_code.zip /content/
!cd /content && unzip -qo FedFIR_GAN_code.zip
%cd /content/paper_pkg
!ls

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/paper_pkg
core.py		make_synthetic.py  README.md	     run_experiments.py
make_report.py	prepare_data.py    requirements.txt


In [ ]:
!pip install -q mne pymatreader tabulate openneuro-py
!python -c "import torch, mne; print('torch', torch.__version__, '| mne', mne.__version__)"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.
torch 2.14.1+cpu | mne 1.13.2


## Step 3. Download the dataset (about 2 GB, 5-15 minutes)
This is the public OpenNeuro dataset **ds004504**. We only need `participants.tsv` and the pre-processed `derivatives` folder.

In [ ]:
import openneuro
openneuro.download(dataset='ds004504', target_dir='/content/ds004504',
                   include=['participants.tsv', 'derivatives'])
!ls /content/ds004504 | head
!ls /content/ds004504/derivatives | head -3

**If the download failed or the `derivatives` folder is empty**, run this alternative cell instead (it uses Amazon S3), then continue:

In [ ]:
# Only run this cell if the previous download did not work
# !pip install -q awscli
# !aws s3 sync --no-sign-request s3://openneuro.org/ds004504 /content/ds004504 --exclude "*" --include "participants.tsv" --include "derivatives/*"

## Step 4. Convert the EEG into training epochs (about 5-10 minutes)
At the end it prints a **DATASET SUMMARY** table. **Copy that table and send it back to Claude**: it is needed for Table 1 of the paper.

In [ ]:
!python prepare_data.py --root /content/ds004504 --out /content/drive/MyDrive/fedfir_work/eeg_epochs.npz 2>&1 | tail -25

## Step 5. Quick test (about 20-40 minutes)
A small run to confirm everything works end to end before you spend hours on the full run. If this finishes without errors and prints `finished`, continue to Step 6.

In [ ]:
!python run_experiments.py --data {WORK}/eeg_epochs.npz --task AD_vs_CN --out {WORK}/test_results --folds 3 --repeats 1 --rounds 8 --local_epochs 1 --central_epochs 8 --gan_iters 100 2>&1 | grep -E "done|finished|Error|Traceback"

## Step 6. The full experiments for the paper (several hours)
Run the two cells below one after the other. They are **resumable**: if Colab disconnects, reconnect, re-run Steps 1 and 2 (skip the download if `/content/ds004504` is gone: the epoch file is already on your Drive), then run this cell again.

Tip: do the AD-vs-CN task first. It is the main result.

In [ ]:
!python run_experiments.py --data {WORK}/eeg_epochs.npz --task AD_vs_CN --out {WORK}/results --dp --sweep 2>&1 | grep -E "done|finished|Error|Traceback"

In [ ]:
!python run_experiments.py --data {WORK}/eeg_epochs.npz --task 3class --out {WORK}/results 2>&1 | grep -E "done|finished|Error|Traceback"

## Step 7. Make the tables and figures

In [ ]:
!python make_report.py --results {WORK}/results --data {WORK}/eeg_epochs.npz --out {WORK}/report

## Step 8. Download your results and send them to Claude
This zips the result files. Download the zip (Files panel on the left, or the cell output) and upload it to Claude with the DATASET SUMMARY from Step 4.

In [ ]:
!cd {WORK} && zip -qr /content/my_results.zip results report
from google.colab import files
files.download('/content/my_results.zip')